# 01 — Extração distribuída do Data Lake

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Tabela:** `ecommerce_itens_pedido`

Lê todas as janelas de tempo real de `ecommerce_itens_pedido` numa única chamada `spark.read`, com glob
no caminho. O `02` e o `03` carregam este notebook via `%run` e recebem o DataFrame `df_itens`.

| # | Regra | Como |
|---|---|---|
| E1 | Leitura distribuída, nada passa pelo driver | `spark.read.options(**adls_options)` com glob `*/*/*/*/` |
| E2 | Erro de acesso aparece na leitura, não depois | `df.schema` dentro do `try` |
| E3 | Cada item sabe de qual lote veio | `dt_lote` extraído de `_metadata.file_path` |
| E4 | Nome da storage account não vira coluna | só o carimbo do caminho é mantido |

In [0]:
%run ./00_setup_config

## 1. Leitura

| Sintoma | Causa provável |
|---|---|
| caminho não encontrado | nenhuma janela de tempo real foi aberta ainda |
| `AuthorizationPermissionMismatch` / 403 | autenticou, mas falta a role *Storage Blob Data Reader* |
| `AADSTS` / `invalid_client` / 401 | credencial errada ou expirada no `.env` |

In [0]:
from pyspark.sql import functions as F


def ler_origem(caminho=CAMINHO_ORIGEM, schema=SCHEMA_ORIGEM):
    try:
        df = spark.read.options(**adls_options).schema(schema).parquet(caminho)
        df.schema
        return df
    except Exception as e:
        msg = str(e)
        if "PATH_NOT_FOUND" in msg or "does not exist" in msg:
            raise FileNotFoundError(f"Nenhum arquivo em {CONTAINER}/{PREFIXO}/: a janela de tempo real pode nao ter sido aberta.") from None
        if "AuthorizationPermissionMismatch" in msg or "403" in msg:
            raise PermissionError("403: o service principal autenticou, mas nao tem leitura no container.") from None
        if "AADSTS" in msg or "invalid_client" in msg or "401" in msg:
            raise PermissionError("401: credencial recusada - conferir ADLS_CLIENT_ID/SECRET/TENANT_ID no .env.") from None
        raise


df_bruto = ler_origem()
print(f"[ok] leitura distribuida: {len(df_bruto.columns)} colunas"); df_bruto.printSchema()

## 2. Linhagem: de qual lote veio cada item

O carimbo do lote está só no caminho do arquivo (`.../AAAA/MM/DD/HHMMSS/...`), em horário de Brasília.
A coluna oculta `_metadata.file_path` expõe esse caminho por linha; o caminho completo traz o nome da
storage account, então só o carimbo vira coluna.

In [0]:
def com_lote(df):
    carimbo = F.regexp_extract(F.col("_metadata.file_path"), r"(\d{4}/\d{2}/\d{2}/\d{6})/", 1)
    return df.withColumn("dt_lote", F.to_timestamp_ntz(carimbo, F.lit("yyyy/MM/dd/HHmmss")))


df_itens = com_lote(df_bruto)
r = df_itens.agg(F.count("*").alias("linhas"), F.countDistinct("dt_lote").alias("lotes"), F.min("dt_lote").alias("primeiro"),
                 F.max("dt_lote").alias("ultimo"), F.sum(F.col("dt_lote").isNull().cast("int")).alias("sem_lote")).first()
print(f"Linhas: {r['linhas']} | lotes: {r['lotes']} | de {r['primeiro']} a {r['ultimo']} | sem carimbo: {r['sem_lote']}")
display(df_itens.limit(10))